In [ ]:
from robovast.common.analysis import read_output_files, read_output_csv

# DATA_DIR is injected by the robovast analysis runner.
DATA_DIR = ''

traj = read_output_files(DATA_DIR, lambda run_dir: read_output_csv(run_dir, "trajectory.csv"))
metrics = read_output_files(DATA_DIR, lambda run_dir: read_output_csv(run_dir, "metrics.csv"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PAD_RADIUS = 0.5

def outcome_of(run, config):
    sel = metrics[(metrics['run'] == run) & (metrics['config'] == config)]
    return sel['outcome'].iloc[0] if len(sel) else 'unknown'

# Static view per run: descent path (x-z) + tilt over time.
for (config, run), g in traj.groupby(['config', 'run']):
    g = g.sort_values('t')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(g['x'], g['z'], '-b')
    ax1.plot(g['x'].iloc[0], g['z'].iloc[0], 'go', label='start')
    ax1.plot(g['x'].iloc[-1], g['z'].iloc[-1], 'rx', label='end')
    ax1.plot([-PAD_RADIUS, PAD_RADIUS], [0, 0], 'g-', lw=4, label='pad')
    ax1.set_xlabel('x [m]'); ax1.set_ylabel('z [m]')
    ax1.set_title(f"{config} run {run}: {outcome_of(run, config)}")
    ax1.legend()
    ax2.plot(g['t'], np.degrees(g['tilt']))
    ax2.axhline(40, color='r', ls='--', lw=0.7); ax2.axhline(-40, color='r', ls='--', lw=0.7)
    ax2.set_xlabel('t [s]'); ax2.set_ylabel('tilt [deg]'); ax2.set_title('tilt')
    plt.tight_layout(); plt.show()

In [ ]:
import os
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

# Animated descent of the first run (saved as descent.gif and embedded inline).
(config, run), g = next(iter(traj.groupby(['config', 'run'])))
g = g.sort_values('t').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(min(g['x'].min(), -PAD_RADIUS) - 1, max(g['x'].max(), PAD_RADIUS) + 1)
ax.set_ylim(0, g['z'].max() + 1)
ax.plot([-PAD_RADIUS, PAD_RADIUS], [0, 0], 'g-', lw=4)
ax.set_xlabel('x [m]'); ax.set_ylabel('z [m]'); ax.set_title(f"{config} run {run}")
path_line, = ax.plot([], [], 'b-', alpha=0.5)
drone, = ax.plot([], [], 'ko', ms=9)

step = max(1, len(g) // 120)
frames = range(0, len(g), step)

def update(i):
    path_line.set_data(g['x'][:i + 1], g['z'][:i + 1])
    drone.set_data([g['x'][i]], [g['z'][i]])
    return path_line, drone

anim = FuncAnimation(fig, update, frames=frames, blit=True)
gif_path = os.path.join(DATA_DIR, 'descent.gif') if DATA_DIR else 'descent.gif'
try:
    anim.save(gif_path, writer=PillowWriter(fps=20))
    print('saved', gif_path)
except Exception as e:  # pillow missing or write-only fs
    print('GIF save skipped:', e)
plt.close(fig)
HTML(anim.to_jshtml())